In [0]:
%sh
pip install gspread pymongo[srv]

In [0]:
%sh pwd

In [0]:
import sys
sys.path.append("/Workspace/Users/matumazparrote@gmail.com/elt_products_scraping")

In [0]:
import json
import os
import pandas as pd
from src.config.mongo_db import mongo_db
from datetime import datetime

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS products;

In [0]:
base_volume = "/Volumes/workspace/products/products_tracker"

In [0]:
mongo_db.connect(dbutils.secrets.get("mondo_db_creds", "MONGO_DB_URI"), db_name="products")

In [0]:
scraped_products_collection = mongo_db.get_database()["scraped_products"]
current_timestamp = datetime.now().strftime("%Y-%m-%d")[0:10]
# current_timestamp = datetime(2026, 5, 20).strftime("%Y-%m-%d")[0:10]
print(f"Timestamp actual: {current_timestamp}")

In [0]:
from bson import json_util
print(f"En total hay {scraped_products_collection.count_documents({})} productos")
scraped_data = list(scraped_products_collection.find({
    "scraped_at": {"$regex": current_timestamp}
}))
print(f"Cantidad: {len(scraped_data)}")
df_scraped_data = pd.json_normalize(scraped_data, max_level=0)
print(df_scraped_data.head())

# Convert ObjectId to string for Parquet compatibility
df_scraped_data['_id'] = df_scraped_data['_id'].astype(str)
# Convert dict to JSON string for Parquet compatibility
df_scraped_data['raw_data'] = df_scraped_data['raw_data'].apply(json.dumps)

year, month, day = current_timestamp.split("-")
path = f"{base_volume}/scraped/year={year}/month={month}/day={day}"
dbutils.fs.mkdirs(path)
# full_path = f"{path}/{current_timestamp}.json"
full_path = f"{path}/{current_timestamp}.parquet"
# with open(full_path, "w", encoding="utf-8") as f:
#     f.write(json_util.dumps(scraped_data, indent=4))
df_scraped_data.to_parquet(full_path, index=False)
print(f"✅ Guardado: {full_path}")